In [1]:
# ============================================================
# Item 1 do Recurso Opcional — Testes Automatizados
# Módulo: API + Suíte de Testes, Célula Única e Autocontida (v2)
# ============================================================
"""
Escreve o arquivo da API (main.py) e a suíte de testes (test_main.py),
e executa os testes — tudo numa única célula, sem depender de nenhuma
execução anterior.

Correção nesta versão: a suíte de testes original revelou um bug real
— uma lista de transações vazia causava erro 500 não tratado (o
TfidfTransformer exige ao menos 1 amostra). Corrigido adicionando
min_length=1 no schema de entrada, rejeitando esse caso com HTTP 422
(erro de validação esperado) em vez de quebrar com erro interno.
"""

import subprocess
import sys
import os


def instalar_dependencias() -> None:
    """Instala fastapi, uvicorn e pytest, caso não estejam disponíveis."""
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "fastapi", "uvicorn", "pytest",
         "scikit-learn", "pandas", "numpy", "joblib", "requests", "--quiet"],
        check=True
    )
    print("✅ Dependências instaladas")


CONTEUDO_MAIN = '''
"""
API de Analise Financeira - FastAPI
Classificacao de transacoes + perfil financeiro + recomendacoes,
seguindo o contrato do edital do Hackathon ONE (Alura + Oracle).
"""

import os
import joblib
import numpy as np
import pandas as pd
import requests
from contextlib import asynccontextmanager
from typing import Literal
from fastapi import FastAPI
from pydantic import BaseModel, Field


class Transacao(BaseModel):
    descricao: str
    valor: float = Field(gt=0, description="Valor deve ser positivo")


class AnaliseFinanceiraRequest(BaseModel):
    renda_mensal: float = Field(gt=0)
    nivel_endividamento: float = Field(ge=0, le=100)
    frequencia_poupanca: Literal["Baixa", "Media", "Alta"]
    transacoes: list[Transacao] = Field(min_length=1, description="Deve haver ao menos uma transacao")


class ClassificarTransacoesRequest(BaseModel):
    transacoes: list[Transacao] = Field(min_length=1, description="Deve haver ao menos uma transacao")


NOMES_EDITAL = {
    "Alimentacao": "alimentacao", "Moradia": "moradia", "Transporte": "transporte",
    "Saude": "saude", "Educacao": "educacao", "Lazer": "entretenimento", "Servicos": "servicos",
}

URLS_OCI = {
    "vetorizador_tfidf.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/hLaYP5zlH3YNL_X1uTG7L6P2V4P_DdFRlTCuydgLM8cFpKGlHPrtj0hqvkLFtNQo/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/vetorizador_tfidf.pkl",
    "modelo_categoria_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/TN6b3FeWkczu1zaQvNyiKqhjivp01Orlz0O28TDmR1wM_V6gZUFKtogP8ixqkfH4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_categoria_producao.pkl",
    "codificador_categorias.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/_9KWGLcV8EKJ-_EiqlX2kZd78GhIO3lVkqBwNYpWfKrNSU-qWuFgGFDAdw6svF42/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_categorias.pkl",
    "modelo_perfil_producao.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/yKaMqbJdI3GNQdB76fIY9pbiMaEfh8l4WE4sVehs5AeY2vTxyLILTv611OVXXc57/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/modelo_perfil_producao.pkl",
    "codificador_perfil.pkl": "https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/DDF588_TNDMhg5v7KBnti9DCqKFd4SI_l3WgyIXkmCJkXUlqCqzJWUnWXSQIHru4/n/grrjqrgpt6r9/b/hackathon-one-g9-team-20/o/models/codificador_perfil.pkl",
}
PASTA_MODELOS = "/content/modelos_api"


def baixar_e_carregar_artefatos() -> dict:
    os.makedirs(PASTA_MODELOS, exist_ok=True)
    for nome_arquivo, url in URLS_OCI.items():
        resposta = requests.get(url)
        resposta.raise_for_status()
        with open(os.path.join(PASTA_MODELOS, nome_arquivo), "wb") as f:
            f.write(resposta.content)
    return {
        "vetorizador_tfidf": joblib.load(os.path.join(PASTA_MODELOS, "vetorizador_tfidf.pkl")),
        "modelo_categoria": joblib.load(os.path.join(PASTA_MODELOS, "modelo_categoria_producao.pkl")),
        "codificador_categorias": joblib.load(os.path.join(PASTA_MODELOS, "codificador_categorias.pkl")),
        "modelo_perfil": joblib.load(os.path.join(PASTA_MODELOS, "modelo_perfil_producao.pkl")),
        "codificador_perfil": joblib.load(os.path.join(PASTA_MODELOS, "codificador_perfil.pkl")),
    }


def classificar_categorias_transacoes(transacoes: list, artefatos: dict) -> list:
    descricoes = [t["descricao"] for t in transacoes]
    vetores = artefatos["vetorizador_tfidf"].transform(descricoes)
    categorias_cod = artefatos["modelo_categoria"].predict(vetores)
    categorias_internas = artefatos["codificador_categorias"].inverse_transform(categorias_cod)
    resultado = []
    for t, cat_interna in zip(transacoes, categorias_internas):
        resultado.append({**t, "categoria": NOMES_EDITAL.get(cat_interna, cat_interna.lower()), "_categoria_interna": cat_interna})
    return resultado


def calcular_resumo_gastos(transacoes_classificadas: list) -> dict:
    categorias_possiveis = list(NOMES_EDITAL.values())
    resumo = {cat: 0.0 for cat in categorias_possiveis}
    for t in transacoes_classificadas:
        resumo[t["categoria"]] += t["valor"]
    return {cat: round(valor, 2) for cat, valor in resumo.items()}


def calcular_resumo_gastos_interno(transacoes_classificadas: list) -> dict:
    categorias_internas = list(NOMES_EDITAL.keys())
    resumo = {cat: 0.0 for cat in categorias_internas}
    for t in transacoes_classificadas:
        resumo[t["_categoria_interna"]] += t["valor"]
    return resumo


def prever_perfil_financeiro(renda_mensal, nivel_endividamento, frequencia_poupanca, resumo_gastos_interno, artefatos) -> tuple:
    mapa_poupanca = {"Baixa": 0, "Media": 1, "Alta": 2}
    comprometimento_gastos = (sum(resumo_gastos_interno.values()) / renda_mensal) * 100
    features = pd.DataFrame([{
        "renda_mensal": renda_mensal, "nivel_endividamento": nivel_endividamento,
        "frequencia_poupanca_cod": mapa_poupanca[frequencia_poupanca],
        "comprometimento_gastos": comprometimento_gastos, **resumo_gastos_interno,
    }])
    probabilidades = artefatos["modelo_perfil"].predict_proba(features)[0]
    indice = np.argmax(probabilidades)
    perfil = artefatos["codificador_perfil"].classes_[indice]
    return perfil, round(float(probabilidades[indice]), 2)


def gerar_recomendacoes(perfil: str, resumo_gastos: dict, frequencia_poupanca: str) -> list:
    if not resumo_gastos:
        return ["Nenhuma transacao informada para gerar recomendacoes especificas."]
    categoria_maior_gasto = max(resumo_gastos, key=resumo_gastos.get)
    if perfil == "Em risco":
        return [
            f"Reduzir gastos com {categoria_maior_gasto}, categoria de maior peso no orcamento",
            "Buscar renegociacao de dividas para reduzir o nivel de endividamento",
        ]
    if perfil == "Em observacao":
        recs = [f"Monitorar gastos recorrentes de {categoria_maior_gasto}"]
        if frequencia_poupanca == "Baixa":
            recs.append("Aumentar a frequencia de poupanca mensal")
        return recs
    return [
        "Manter o padrao atual de organizacao financeira",
        "Considerar investir o excedente mensal para objetivos de longo prazo",
    ]


def analisar_financas(dados_entrada: dict, artefatos: dict) -> dict:
    transacoes_classificadas = classificar_categorias_transacoes(dados_entrada["transacoes"], artefatos)
    resumo_gastos_edital = calcular_resumo_gastos(transacoes_classificadas)
    resumo_gastos_interno = calcular_resumo_gastos_interno(transacoes_classificadas)
    perfil, probabilidade = prever_perfil_financeiro(
        dados_entrada["renda_mensal"], dados_entrada["nivel_endividamento"],
        dados_entrada["frequencia_poupanca"], resumo_gastos_interno, artefatos
    )
    recomendacoes = gerar_recomendacoes(perfil, resumo_gastos_edital, dados_entrada["frequencia_poupanca"])
    return {
        "perfil_financeiro": perfil, "probabilidade": probabilidade,
        "resumo_gastos": {k: v for k, v in resumo_gastos_edital.items() if v > 0},
        "recomendacoes": recomendacoes,
    }


artefatos_globais = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    print("Carregando modelos do OCI Object Storage...")
    artefatos_globais.update(baixar_e_carregar_artefatos())
    print("Modelos carregados com sucesso")
    yield
    artefatos_globais.clear()


app = FastAPI(title="API de Analise Financeira - G9 Team 20", lifespan=lifespan)


@app.get("/")
def raiz():
    return {"status": "API no ar", "modelos_carregados": len(artefatos_globais) > 0}


@app.post("/analise-financeira")
def analise_financeira(dados: AnaliseFinanceiraRequest):
    return analisar_financas(dados.model_dump(), artefatos_globais)


@app.post("/classificar-transacoes")
def classificar_transacoes(dados: ClassificarTransacoesRequest):
    transacoes_dict = [t.model_dump() for t in dados.transacoes]
    transacoes_classificadas = classificar_categorias_transacoes(transacoes_dict, artefatos_globais)
    return {"transacoes_classificadas": [
        {"descricao": t["descricao"], "valor": t["valor"], "categoria": t["categoria"]}
        for t in transacoes_classificadas
    ]}
'''


CONTEUDO_TESTES = '''
"""Suite de testes automatizados da API de Analise Financeira."""
import sys
sys.path.insert(0, "/content")
import pytest
from fastapi.testclient import TestClient
from main import app


@pytest.fixture(scope="module")
def cliente():
    with TestClient(app) as c:
        yield c


class TestHealthCheck:
    def test_status_no_ar(self, cliente):
        resposta = cliente.get("/")
        assert resposta.status_code == 200
        assert resposta.json()["status"] == "API no ar"

    def test_modelos_carregados(self, cliente):
        resposta = cliente.get("/")
        assert resposta.json()["modelos_carregados"] is True


class TestAnaliseFinanceira:
    def test_perfil_saudavel(self, cliente):
        dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Supermercado", "valor": 420}, {"descricao": "Combustivel", "valor": 300}, {"descricao": "Streaming", "valor": 40}]}
        resposta = cliente.post("/analise-financeira", json=dados)
        assert resposta.status_code == 200
        assert resposta.json()["perfil_financeiro"] == "Saudavel"

    def test_perfil_em_risco(self, cliente):
        dados = {"renda_mensal": 3000, "nivel_endividamento": 62, "frequencia_poupanca": "Baixa",
            "transacoes": [{"descricao": "Aluguel Residencial", "valor": 1200}, {"descricao": "Posto Ipiranga", "valor": 250}]}
        resposta = cliente.post("/analise-financeira", json=dados)
        assert resposta.status_code == 200
        assert resposta.json()["perfil_financeiro"] == "Em risco"

    def test_perfil_em_observacao(self, cliente):
        dados = {"renda_mensal": 5200, "nivel_endividamento": 38, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Mercado Atacadao", "valor": 380}, {"descricao": "Aplicativo Uber", "valor": 150}]}
        resposta = cliente.post("/analise-financeira", json=dados)
        assert resposta.status_code == 200
        assert resposta.json()["perfil_financeiro"] == "Em observacao"

    def test_resposta_contem_campos_obrigatorios(self, cliente):
        dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Supermercado", "valor": 420}]}
        resposta = cliente.post("/analise-financeira", json=dados)
        corpo = resposta.json()
        assert "perfil_financeiro" in corpo and "probabilidade" in corpo
        assert "resumo_gastos" in corpo and "recomendacoes" in corpo

    def test_nomenclatura_categorias_formato_edital(self, cliente):
        dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Streaming", "valor": 40}]}
        resposta = cliente.post("/analise-financeira", json=dados)
        categorias = resposta.json()["resumo_gastos"].keys()
        assert all(cat.islower() for cat in categorias)


class TestValidacaoDeEntrada:
    def test_valor_negativo_rejeitado(self, cliente):
        dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Teste", "valor": -50}]}
        assert cliente.post("/analise-financeira", json=dados).status_code == 422

    def test_renda_zero_rejeitada(self, cliente):
        dados = {"renda_mensal": 0, "nivel_endividamento": 25, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Teste", "valor": 50}]}
        assert cliente.post("/analise-financeira", json=dados).status_code == 422

    def test_endividamento_acima_de_100_rejeitado(self, cliente):
        dados = {"renda_mensal": 4500, "nivel_endividamento": 150, "frequencia_poupanca": "Media",
            "transacoes": [{"descricao": "Teste", "valor": 50}]}
        assert cliente.post("/analise-financeira", json=dados).status_code == 422

    def test_frequencia_poupanca_invalida_rejeitada(self, cliente):
        dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Nenhuma",
            "transacoes": [{"descricao": "Teste", "valor": 50}]}
        assert cliente.post("/analise-financeira", json=dados).status_code == 422

    def test_lista_transacoes_vazia_e_rejeitada(self, cliente):
        """Corrigido: lista vazia agora e rejeitada (min_length=1), nao mais aceita.

        Descoberto por esta mesma suite de testes: uma lista vazia causava
        erro 500 nao tratado no TfidfTransformer, que exige ao menos 1
        amostra. A correcao move essa rejeicao para a camada de validacao
        (HTTP 422), um erro esperado e documentado, em vez de uma falha
        interna do servidor.
        """
        dados = {"renda_mensal": 4500, "nivel_endividamento": 25, "frequencia_poupanca": "Media", "transacoes": []}
        assert cliente.post("/analise-financeira", json=dados).status_code == 422


class TestClassificarTransacoes:
    def test_classifica_transacoes_corretamente(self, cliente):
        dados = {"transacoes": [{"descricao": "Netflix", "valor": 40}, {"descricao": "Posto Ipiranga", "valor": 200}]}
        resposta = cliente.post("/classificar-transacoes", json=dados)
        assert resposta.status_code == 200
        corpo = resposta.json()
        assert len(corpo["transacoes_classificadas"]) == 2
        assert all("categoria" in t for t in corpo["transacoes_classificadas"])

    def test_nao_exige_dados_de_perfil(self, cliente):
        dados = {"transacoes": [{"descricao": "Farmacia Pague Menos", "valor": 80}]}
        assert cliente.post("/classificar-transacoes", json=dados).status_code == 200

    def test_lista_vazia_rejeitada(self, cliente):
        """Mesma correcao aplicada ao endpoint dedicado de classificacao."""
        dados = {"transacoes": []}
        assert cliente.post("/classificar-transacoes", json=dados).status_code == 422
'''


def escrever_arquivos() -> None:
    """Grava main.py e test_main.py em disco, garantindo estado limpo mesmo após reinício de sessão."""
    with open("/content/main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_MAIN)
    with open("/content/test_main.py", "w", encoding="utf-8") as f:
        f.write(CONTEUDO_TESTES)
    print("✅ main.py e test_main.py escritos em /content")


def executar_suite_de_testes() -> None:
    """Executa a suíte via pytest, com cwd fixado em /content para evitar ModuleNotFoundError."""
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "test_main.py", "-v"],
        capture_output=True, text=True, cwd="/content"
    )
    print(resultado.stdout)
    if resultado.stderr:
        print("⚠️ Saída de erro/avisos:")
        print(resultado.stderr)


# ------------------------------------------------------------
# EXECUÇÃO
# ------------------------------------------------------------
instalar_dependencias()
escrever_arquivos()
executar_suite_de_testes()

✅ Dependências instaladas
✅ main.py e test_main.py escritos em /content
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.14.2, langsmith-0.10.2, typeguard-4.5.2
collecting ... collected 15 items

test_main.py::TestHealthCheck::test_status_no_ar PASSED                  [  6%]
test_main.py::TestHealthCheck::test_modelos_carregados PASSED            [ 13%]
test_main.py::TestAnaliseFinanceira::test_perfil_saudavel PASSED         [ 20%]
test_main.py::TestAnaliseFinanceira::test_perfil_em_risco PASSED         [ 26%]
test_main.py::TestAnaliseFinanceira::test_perfil_em_observacao PASSED    [ 33%]
test_main.py::TestAnaliseFinanceira::test_resposta_contem_campos_obrigatorios PASSED [ 40%]
test_main.py::TestAnaliseFinanceira::test_nomenclatura_categorias_formato_edital PASSED [ 46%]
test_main.py::TestValidacaoDeEntrada::test_va